# 07 - Results Interpretation

This notebook consolidates the social-network and forecasting outputs around the research question:

> Can social interaction patterns between Yelp users help predict future business review activity beyond historical review trends alone?

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "new_orleans"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

METRICS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_metrics.csv"
PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_predictions.csv"
GRAPH_SUMMARY_PATH = PROCESSED_DIR / "social_graph_summary.json"
FEATURE_SUMMARY_PATH = PROCESSED_DIR / "forecasting_feature_summary.json"

metrics = pd.read_csv(METRICS_OUTPUT_PATH)
predictions = pd.read_csv(PREDICTIONS_OUTPUT_PATH)
with GRAPH_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    graph_summary = json.load(file)
with FEATURE_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    feature_summary = json.load(file)

metrics.sort_values(["split", "WAPE", "MAE"])

,split,model,rows,MAE,RMSE,WAPE
0,primary_covid_test,Baseline: last month,27624,1.311323,2.653812,0.688473
1,primary_covid_test,Baseline: rolling 3-month avg,27624,1.342685,2.954533,0.704938
2,primary_covid_test,ML: historical,27624,1.445322,3.001475,0.758825
3,primary_covid_test,ML: historical + business + SNA,27624,1.477177,3.054916,0.775550
4,primary_covid_test,ML: historical + business,27624,1.479147,3.074246,0.776584
5,primary_covid_test,Baseline: seasonal naive,27624,2.877643,6.556732,1.510824
6,secondary_pre_covid_test,ML: historical + business,13812,1.832783,3.116894,0.383924
7,secondary_pre_covid_test,ML: historical + business + SNA,13812,1.837254,3.112236,0.384860
8,secondary_pre_covid_test,ML: historical,13812,1.889936,3.157809,0.395896
9,secondary_pre_covid_test,Baseline: rolling 3-month avg,13812,1.954798,3.306649,0.409483


In [2]:
summary_rows = []
for split_name, split_metrics in metrics.groupby("split"):
    ranked = split_metrics.sort_values("WAPE").reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    hist_business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + business + SNA"].iloc[0]
    summary_rows.append({
        "split": split_name,
        "best_model": best["model"],
        "best_WAPE": best["WAPE"],
        "historical_WAPE": hist["WAPE"],
        "historical_business_WAPE": hist_business["WAPE"],
        "sna_WAPE": sna["WAPE"],
        "sna_vs_historical_business_relative_change": (sna["WAPE"] - hist_business["WAPE"]) / hist_business["WAPE"],
        "sna_vs_historical_relative_change": (sna["WAPE"] - hist["WAPE"]) / hist["WAPE"],
    })
interpretation_summary = pd.DataFrame(summary_rows)
interpretation_summary

,split,best_model,best_WAPE,historical_WAPE,historical_business_WAPE,sna_WAPE,sna_vs_historical_business_relative_change,sna_vs_historical_relative_change
0,primary_covid_test,Baseline: last month,0.688473,0.758825,0.776584,0.77555,-0.001332,0.022040
1,secondary_pre_covid_test,ML: historical + business,0.383924,0.395896,0.383924,0.38486,0.002440,-0.027875


In [3]:
print("Social graph summary")
for key, value in graph_summary.items():
    print(f"{key}: {value}")

print("\nForecasting dataset summary")
for key, value in feature_summary.items():
    print(f"{key}: {value}")

Social graph summary
active_review_threshold: 5
threshold_candidates: [2, 3, 5, 10, 20]
edge_weight_formula: 1 + log1p(shared_business_count) + category_jaccard
reviewing_users: 245421
matched_user_profiles: 245419
active_users: 26598
graph_nodes: 26598
graph_edges: 116558
mean_edge_weight: 1.833072733525831
mean_edge_shared_business_count: 2.048550936014688
mean_edge_category_jaccard: 0.2671618532172656
connected_components: 11508
largest_component_size: 14965
isolated_active_users: 11387
community_method: weighted_louvain_largest_component
communities_assigned: 82
threshold_sensitivity_output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\active_reviewer_threshold_sensitivity.csv
output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\user_network_features.csv

Forecasting dataset summary
min_total_reviews: 100
min_active_months: 36
business_count: 1151
row_count: 95533
feature_month_min: 2015-01
featur

In [4]:
for _, row in interpretation_summary.iterrows():
    split = row["split"]
    change_vs_business = row["sna_vs_historical_business_relative_change"] * 100
    change_vs_hist = row["sna_vs_historical_relative_change"] * 100
    direction_business = "improved" if change_vs_business < 0 else "worsened"
    direction_hist = "improved" if change_vs_hist < 0 else "worsened"
    print(f"{split}:")
    print(f"  Best model: {row['best_model']} with WAPE={row['best_WAPE']:.4f}")
    print(f"  SNA model {direction_business} WAPE vs historical+business by {abs(change_vs_business):.2f}%")
    print(f"  SNA model {direction_hist} WAPE vs historical-only by {abs(change_vs_hist):.2f}%")

primary_covid_test:
  Best model: Baseline: last month with WAPE=0.6885
  SNA model improved WAPE vs historical+business by 0.13%
  SNA model worsened WAPE vs historical-only by 2.20%
secondary_pre_covid_test:
  Best model: ML: historical + business with WAPE=0.3839
  SNA model worsened WAPE vs historical+business by 0.24%
  SNA model improved WAPE vs historical-only by 2.79%


## Interpretation Framework

Results are interpreted in three layers:

1. **Forecastability:** how far simple temporal baselines go.
2. **Business metadata value:** whether category and static business attributes add signal.
3. **SNA value:** whether reviewer-network structure improves over historical and business-only features.

The SNA layer now separates generic friendship connectivity from locally weighted social exposure. This makes the experiment more faithful to the project objective: testing whether community structure, not just activity volume, helps forecast local Yelp attention.

The weighted SNA result is mixed but informative. It is marginally better than historical + business in the COVID-era split, but the last-month baseline still wins by a wide margin. In the pre-COVID split, weighted SNA improves over historical-only but does not beat historical + business. The fair interpretation is that SNA is measurable and methodologically meaningful, but not yet a strong incremental predictor for raw one-month-ahead review counts.

## Limitations

- Yelp friendship links are static; friendship formation dates are unavailable.
- Network features describe social context, not causal influence.
- The COVID-era test period includes a major external shock.
- The activity threshold improves reliability but narrows the cohort to active businesses.
- Review count measures Yelp engagement, not revenue or true customer volume.
- Static business metadata may include end-of-dataset information.

## Final Academic Position

The project is best presented as an interpretable multimodal forecasting experiment about **local attention dynamics**. It successfully combines temporal behavior, business metadata, and weighted social-network structure. The current evidence does **not** show a strong or consistent forecasting gain from SNA features for raw review-count regression.

That negative result is useful: it shows one challenge organisations face when using diverse data modalities. A richer data modality can be expensive to prepare and theoretically relevant, while still adding limited predictive value for a specific target. The next defensible step is to test SNA on a target where social exposure should matter more, such as attention-pulse or trending-business prediction.